In [0]:
storage_account_name = "aparnastshellenergyuks"
storage_account_key = dbutils.secrets.get(
    scope="aparna-kv-shellenergy-scope",
    key="aparna-adls-storage-account-key"
)
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/batch"
gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/batch"

df_customers = spark.read.format("delta").load(f"{silver_path}/customers/")
df_sites = spark.read.format("delta").load(f"{silver_path}/sites/")
df_contracts = spark.read.format("delta").load(f"{silver_path}/contracts/")
df_prices = spark.read.format("delta").load(f"{silver_path}/wholesale_prices/")
df_readings = spark.read.format("delta").load(f"{silver_path}/meter_readings/")
df_invoices = spark.read.format("delta").load(f"{silver_path}/billing_invoices/")

print("Silver row counts:")
for name, df in [("customers", df_customers), ("sites", df_sites), 
                  ("contracts", df_contracts), ("wholesale_prices", df_prices),
                  ("meter_readings", df_readings), ("billing_invoices", df_invoices)]:
    print(f"  {name}: {df.count():,}")

In [0]:
from pyspark.sql.functions import (
    col, sum as spark_sum, avg, year, month, date_trunc, round as spark_round,
    when, lit
)

df_monthly_consumption = (
    df_readings
    .withColumn("billing_month", date_trunc("month", col("interval_start")))
    .groupBy("site_id", "billing_month")
    .agg(
        spark_sum("kwh_interval").alias("actual_kwh"),
        spark_sum(when(col("data_quality_flag") == "GAP_FILLED", col("kwh_interval"))).alias("gap_filled_kwh"),
        spark_sum(when(col("data_quality_flag") == "OUTAGE", 1).otherwise(0)).alias("outage_interval_count")
    )
)

print(f"Monthly consumption rows: {df_monthly_consumption.count()}")
display(df_monthly_consumption.orderBy("site_id", "billing_month").limit(10))

In [0]:
from pyspark.sql.functions import months_between, datediff, lit

# Simplify: take each contract's annual_kwh and convert to a flat monthly figure 
# (annual / 12) - a reasonable approximation given most contracts run full months 
# in our dataset, and avoids the complexity of mid-month proration for this analysis
df_contracts_monthly = (
    df_contracts
    .withColumn("contracted_monthly_kwh", col("contracted_annual_kwh") / 12)
    .select("site_id", "contract_id", "start_date", "end_date", 
            "contracted_monthly_kwh", "rate_gbp_per_kwh", "standing_charge_gbp_per_day")
)

display(df_contracts_monthly.orderBy("site_id", "start_date").limit(10))

In [0]:
from pyspark.sql.functions import col, lit

# Join consumption (site, billing_month) against contracts where the billing_month
# falls within the contract's [start_date, end_date] window
df_with_contract = (
    df_monthly_consumption.alias("mc")
    .join(
        df_contracts_monthly.alias("c"),
        (col("mc.site_id") == col("c.site_id")) &
        (col("mc.billing_month") >= col("c.start_date")) &
        (col("mc.billing_month") <= col("c.end_date")),
        how="left"
    )
    .select(
        col("mc.site_id"),
        col("mc.billing_month"),
        col("mc.actual_kwh"),
        col("mc.gap_filled_kwh"),
        col("mc.outage_interval_count"),
        col("c.contract_id"),
        col("c.contracted_monthly_kwh"),
        col("c.rate_gbp_per_kwh"),
        col("c.standing_charge_gbp_per_day")
    )
)

print(f"Rows after contract join: {df_with_contract.count()}")

# Sanity check: any site-months that matched ZERO contracts (would show as nulls)?
unmatched = df_with_contract.filter(col("contract_id").isNull()).count()
print(f"Site-months with NO matching contract: {unmatched}")

display(df_with_contract.orderBy("site_id", "billing_month").limit(10))

In [0]:
from pyspark.sql.functions import avg, col, round as spark_round, datediff, last_day

# Average wholesale price (£/MWh) per billing month, to estimate "true" procurement cost
df_monthly_wholesale_avg = (
    df_prices
    .withColumn("billing_month", date_trunc("month", col("interval_start")))
    .groupBy("billing_month")
    .agg(avg("price_gbp_per_mwh").alias("avg_wholesale_price_gbp_per_mwh"))
)

display(df_monthly_wholesale_avg.orderBy("billing_month"))

In [0]:
# Join wholesale price onto the contract-joined data, then calculate the business metrics
df_gold_site_monthly = (
    df_with_contract
    .join(df_monthly_wholesale_avg, on="billing_month", how="left")
    .withColumn("days_in_month", datediff(last_day(col("billing_month")), col("billing_month")) + 1)
    .withColumn(
        "variance_pct",
        spark_round(((col("actual_kwh") - col("contracted_monthly_kwh")) / col("contracted_monthly_kwh")) * 100, 2)
    )
    .withColumn(
        "true_wholesale_cost_gbp",
        # actual_kwh is in kWh, wholesale price is per MWh -> divide by 1000
        spark_round((col("actual_kwh") / 1000) * col("avg_wholesale_price_gbp_per_mwh"), 2)
    )
    .withColumn(
        "billed_energy_charge_gbp",
        spark_round(col("actual_kwh") * col("rate_gbp_per_kwh"), 2)
    )
    .withColumn(
        "billed_standing_charge_gbp",
        spark_round(col("standing_charge_gbp_per_day") * col("days_in_month"), 2)
    )
    .withColumn(
        "billed_total_gbp",
        spark_round(col("billed_energy_charge_gbp") + col("billed_standing_charge_gbp"), 2)
    )
    .withColumn(
        "margin_gbp",
        spark_round(col("billed_total_gbp") - col("true_wholesale_cost_gbp"), 2)
    )
    .withColumn(
        "margin_pct",
        spark_round((col("margin_gbp") / col("billed_total_gbp")) * 100, 2)
    )
)

print(f"gold_site_monthly_summary row count: {df_gold_site_monthly.count()}")
display(df_gold_site_monthly.orderBy("site_id", "billing_month").select(
    "site_id", "billing_month", "actual_kwh", "contracted_monthly_kwh", "variance_pct",
    "true_wholesale_cost_gbp", "billed_total_gbp", "margin_gbp", "margin_pct"
).limit(10))

In [0]:
'''
Adds a modeled UK C&I bill cost stack (network charges, policy costs, supplier 
operating costs) on top of the wholesale cost calculation, since billed revenue 
minus wholesale cost alone overstates margin - most of that gap is pass-through 
cost to National Grid/DNOs and government schemes, not Shell's actual profit.
'''

from pyspark.sql.functions import col, round as spark_round

# Modeled UK C&I bill cost stack (industry-typical proportions, applied to billed_total_gbp)
NETWORK_CHARGE_PCT = 0.27       # transmission + distribution charges - pass-through to National Grid/DNOs
POLICY_COST_PCT = 0.12          # environmental/policy obligations - pass-through to government schemes
OPERATING_COST_PCT = 0.08       # Shell's own operating costs - real cost, but not profit

df_gold_site_monthly = (
    df_gold_site_monthly
    .withColumn("network_charges_gbp", spark_round(col("billed_total_gbp") * NETWORK_CHARGE_PCT, 2))
    .withColumn("policy_costs_gbp", spark_round(col("billed_total_gbp") * POLICY_COST_PCT, 2))
    .withColumn("operating_costs_gbp", spark_round(col("billed_total_gbp") * OPERATING_COST_PCT, 2))
    .withColumn(
        "net_margin_gbp",
        spark_round(
            col("billed_total_gbp") - col("true_wholesale_cost_gbp") 
            - col("network_charges_gbp") - col("policy_costs_gbp") - col("operating_costs_gbp"),
            2
        )
    )
    .withColumn(
        "net_margin_pct",
        spark_round((col("net_margin_gbp") / col("billed_total_gbp")) * 100, 2)
    )
    # Keep the old gross figure too, clearly relabeled - useful to show both in the dashboard
    .withColumnRenamed("margin_gbp", "gross_margin_over_wholesale_gbp")
    .withColumnRenamed("margin_pct", "gross_margin_over_wholesale_pct")
)

display(df_gold_site_monthly.orderBy("site_id", "billing_month").select(
    "site_id", "billing_month", "billed_total_gbp", "true_wholesale_cost_gbp",
    "network_charges_gbp", "policy_costs_gbp", "operating_costs_gbp",
    "net_margin_gbp", "net_margin_pct"
).limit(10))

In [0]:
df_gold_site_monthly.write.format("delta").mode("overwrite").save(f"{gold_path}/site_monthly_summary/")

# Verify
df_check = spark.read.format("delta").load(f"{gold_path}/site_monthly_summary/")
print(f"Verified row count: {df_check.count()}")
display(df_check.orderBy("site_id", "billing_month").limit(5))

In [0]:
from pyspark.sql.functions import to_date, col, sum as spark_sum, avg, round as spark_round

# Daily actual consumption per site
df_daily_consumption = (
    df_readings
    .withColumn("reading_date", to_date(col("interval_start")))
    .groupBy("site_id", "reading_date")
    .agg(spark_sum("kwh_interval").alias("actual_daily_kwh"))
)

# Daily average wholesale price
df_daily_wholesale_avg = (
    df_prices
    .withColumn("reading_date", to_date(col("interval_start")))
    .groupBy("reading_date")
    .agg(avg("price_gbp_per_mwh").alias("avg_daily_wholesale_price_gbp_per_mwh"))
)

print(f"Daily consumption rows: {df_daily_consumption.count()}")
print(f"Daily wholesale price rows: {df_daily_wholesale_avg.count()}")

In [0]:
from pyspark.sql.functions import col, lit

# Convert each contract's monthly figure into a flat daily run-rate (same 
# simplification as before - monthly/days_in_month, not precise mid-month proration)
df_contracts_daily_rate = (
    df_contracts
    .withColumn("contracted_daily_kwh", col("contracted_annual_kwh") / 365.25)
    .select("site_id", "contract_id", "start_date", "end_date", "contracted_daily_kwh")
)

# Match each site-day to its active contract (same overlap logic as the monthly table)
df_daily_with_contract = (
    df_daily_consumption.alias("dc")
    .join(
        df_contracts_daily_rate.alias("c"),
        (col("dc.site_id") == col("c.site_id")) &
        (col("dc.reading_date") >= col("c.start_date")) &
        (col("dc.reading_date") <= col("c.end_date")),
        how="left"
    )
    .select(
        col("dc.site_id"), col("dc.reading_date"), col("dc.actual_daily_kwh"),
        col("c.contract_id"), col("c.contracted_daily_kwh")
    )
)

unmatched = df_daily_with_contract.filter(col("contract_id").isNull()).count()
print(f"Site-days with NO matching contract: {unmatched}")
print(f"Rows after contract join: {df_daily_with_contract.count()}")

In [0]:
from pyspark.sql.functions import col, round as spark_round, abs as spark_abs

df_gold_exposure_daily = (
    df_daily_with_contract
    .join(df_daily_wholesale_avg, on="reading_date", how="left")
    .withColumn(
        "variance_kwh",
        spark_round(col("actual_daily_kwh") - col("contracted_daily_kwh"), 2)
    )
    .withColumn(
        "variance_pct",
        spark_round((col("variance_kwh") / col("contracted_daily_kwh")) * 100, 2)
    )
    .withColumn(
        # The core risk number: the £ cost/benefit of that day's volume deviation,
        # valued at that day's actual wholesale price. Positive = consumed more 
        # than contracted, exposed to that day's price on the EXTRA volume.
        # Negative = consumed less, i.e. paid for capacity not used.
        "daily_exposure_gbp",
        spark_round((col("variance_kwh") / 1000) * col("avg_daily_wholesale_price_gbp_per_mwh"), 2)
    )
    .withColumn(
        "exposure_direction",
        when(col("variance_kwh") > 0, "OVER_CONSUMPTION")
        .when(col("variance_kwh") < 0, "UNDER_CONSUMPTION")
        .otherwise("ON_TARGET")
    )
)

print(f"gold_exposure_daily row count: {df_gold_exposure_daily.count()}")
display(df_gold_exposure_daily.orderBy("site_id", "reading_date").select(
    "site_id", "reading_date", "actual_daily_kwh", "contracted_daily_kwh",
    "variance_pct", "daily_exposure_gbp", "exposure_direction"
).limit(10))

# Sanity check: distribution of exposure direction across all site-days
display(df_gold_exposure_daily.groupBy("exposure_direction").count())

In [0]:
from pyspark.sql.functions import min as spark_min, max as spark_max, avg as spark_avg, stddev

display(df_gold_exposure_daily.select(
    spark_min("daily_exposure_gbp").alias("min_exposure"),
    spark_max("daily_exposure_gbp").alias("max_exposure"),
    spark_avg("daily_exposure_gbp").alias("avg_exposure"),
    stddev("daily_exposure_gbp").alias("stddev_exposure")
))

In [0]:
df_gold_exposure_daily.write.format("delta").mode("overwrite").save(f"{gold_path}/exposure_daily/")

# Verify
df_check = spark.read.format("delta").load(f"{gold_path}/exposure_daily/")
print(f"Verified row count: {df_check.count()}")

In [0]:
from pyspark.sql.functions import sum as spark_sum, count, when, col, round as spark_round

df_gold_portfolio_summary = (
    df_gold_site_monthly
    .groupBy("billing_month")
    .agg(
        count("site_id").alias("active_site_count"),
        spark_sum("actual_kwh").alias("total_actual_kwh"),
        spark_sum("contracted_monthly_kwh").alias("total_contracted_kwh"),
        spark_sum("billed_total_gbp").alias("total_billed_gbp"),
        spark_sum("true_wholesale_cost_gbp").alias("total_wholesale_cost_gbp"),
        spark_sum("net_margin_gbp").alias("total_net_margin_gbp"),
        spark_sum(when(col("net_margin_gbp") < 0, 1).otherwise(0)).alias("lossmaking_site_count"),
        spark_sum(when(col("variance_pct") > 10, 1).otherwise(0)).alias("sites_over_10pct_variance"),
    )
    .withColumn(
        "portfolio_variance_pct",
        spark_round(((col("total_actual_kwh") - col("total_contracted_kwh")) / col("total_contracted_kwh")) * 100, 2)
    )
    .withColumn(
        "portfolio_net_margin_pct",
        spark_round((col("total_net_margin_gbp") / col("total_billed_gbp")) * 100, 2)
    )
)

print(f"gold_portfolio_summary row count: {df_gold_portfolio_summary.count()}")
display(df_gold_portfolio_summary.orderBy("billing_month"))

In [0]:
df_gold_portfolio_summary.write.format("delta").mode("overwrite").save(f"{gold_path}/portfolio_summary/")

# Verify
df_check = spark.read.format("delta").load(f"{gold_path}/portfolio_summary/")
print(f"Verified row count: {df_check.count()}")
display(df_check.orderBy("billing_month"))